# Databehandling

## 1. Opsætning

In [1]:
%pip install pandas openpyxl xlrd

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, re
import numpy as np
import pandas as pd
from functools import reduce

BASE = os.getcwd()
RAW_DIR = os.path.join(BASE, "raw")
CLEAN_DIR = os.path.join(BASE, "clean")
os.makedirs(CLEAN_DIR, exist_ok=True)

START = "2015-01-31"   # første måned (månedsslut)
SLUT  = "2026-09-30"   # fælles slutmåned

# Filnavne i raw-mappen – ret her, hvis jeres filer hedder noget andet
FILER = {
    "bbg_m":   "bloomberg_indices_monthly.xlsx",
    "bbg_w":   "bloomberg_weekly.xlsx",
    "GPR":     "GPR_data.xls",
    "TPU":     "TPU_data.xlsx",
    "VIXCLS":     "VIXCLS.csv",
    "USD_INDEX":     "DTWEXBGS.csv",
    "DGS10":   "DGS10.csv",
    "DGS2":   "DGS2.csv",
    "CPI":     "CPILFESL.csv",
    "INDPRO":  "INDPRO.csv",
    "NFP":     "PAYEMS.csv",
    "DNFPVALE":     "Nationalbanken_DNFPVALE.csv",
    "Toldsatser":    "Harmonized System District-level Imports.csv",
    "crsp_eq": "us_equity_returns_crsp_monthly.csv",
    "crsp_bd": "us_treasury_returns_crsp_monthly.csv",
}

## 2. Hjælpefunktioner

In [3]:
def raw(navn):
    return os.path.join(RAW_DIR, FILER[navn])

def til_md(s):
    """Dato -> månedsslut, så alle filer har samme nøgle."""
    return pd.to_datetime(s).dt.to_period("M").dt.to_timestamp("M")

def afgraens(df, kol="month"):
    return df[(df[kol] >= START) & (df[kol] <= SLUT)].reset_index(drop=True)

def gem(df, fil):
    df.to_csv(os.path.join(CLEAN_DIR, fil), index=False)
    print(f"Gemt {fil}: {df.shape}, {df['month'].min().date()} -> {df['month'].max().date()}")

def les_fred(navn, kol):
    """Læs FRED-csv. Klarer både FRED's format (komma, ÅÅÅÅ-MM-DD)
    og en fil gemt i dansk Excel (semikolon, decimalkomma, DD.MM.ÅÅÅÅ)."""
    df = pd.read_csv(raw(navn), sep=None, engine="python", encoding="latin-1")
    df[kol] = pd.to_numeric(df[kol].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    dato = df["observation_date"].astype(str)
    fmt = "%d.%m.%Y" if dato.str.contains(r"\.").any() else "%Y-%m-%d"
    df["date"] = pd.to_datetime(dato, format=fmt)
    return df[["date", kol]]

def les_bbg_ark(fil, ark, freq):
    """Ét Bloomberg-ark: kolonne 0 = dato, kolonne 1 = værdi.
    Er filen gemt uden Bloomberg, står der '#NAME?' i stedet for første dato
    (cellen med BDH-formlen). Værdien er korrekt, så datoen genskabes ud fra næste række."""
    d = pd.read_excel(fil, sheet_name=ark, header=None).iloc[:, :2]
    d.columns = ["date", "value"]
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d["value"] = pd.to_numeric(d["value"], errors="coerce")
    if pd.isna(d.loc[0, "date"]) and pd.notna(d.loc[0, "value"]):
        if freq == "M":
            d.loc[0, "date"] = (d.loc[1, "date"].to_period("M") - 1).to_timestamp()   # kun måneden bruges
        elif freq == "W":
            d.loc[0, "date"] = d.loc[1, "date"] - pd.Timedelta(days=7)
        else:
            print(f"  ADVARSEL {ark}: første dato mangler og kan ikke genskabes – rækken droppes")
        if freq in ("M", "W"):
            print(f"  {ark}: første dato manglede ('#NAME?') – genskabt som {d.loc[0, 'date'].date()}")
    return d.dropna(subset=["date"]).rename(columns={"value": ark})

## 3. Bloomberg – månedlig

In [4]:
fil = raw("bbg_m")
frames = []
for ark in pd.ExcelFile(fil).sheet_names:
    d = les_bbg_ark(fil, ark, "M")
    d["month"] = til_md(d["date"])
    frames.append(d[["month", ark]])

bbg = reduce(lambda l, r: pd.merge(l, r, on="month", how="outer"), frames)
bbg = bbg.sort_values("month").reset_index(drop=True)
bbg.columns = [c.replace(" Index", "") for c in bbg.columns]
bbg = bbg.rename(columns={"MXWOH": "MSCIW_H"})        # hvis arket stadig hedder MXWOH
bbg = afgraens(bbg)

assert bbg["month"].is_unique
print(bbg.head(2))
print("Manglende værdier:\n", bbg.isna().sum()[lambda s: s > 0])   # forventet: kun basis swap før feb. 2021
gem(bbg, "bloomberg_monthly.csv")

  EURUSD: første dato manglede ('#NAME?') – genskabt som 2015-01-01
  USDDKK: første dato manglede ('#NAME?') – genskabt som 2015-01-01
  EUR3M: første dato manglede ('#NAME?') – genskabt som 2015-01-01
  USD OIS 3m: første dato manglede ('#NAME?') – genskabt som 2015-01-01
  EUR OIS 3m: første dato manglede ('#NAME?') – genskabt som 2015-01-01
  EURUSD basis swap 3m: første dato manglede ('#NAME?') – genskabt som 2021-02-01
  SPXT: første dato manglede ('#NAME?') – genskabt som 2015-01-01
  LUATTRUU: første dato manglede ('#NAME?') – genskabt som 2015-01-01
  MSCIW_H: første dato manglede ('#NAME?') – genskabt som 2015-01-01
  GLTREAS_H: første dato manglede ('#NAME?') – genskabt som 2015-01-01
       month  EURUSD  USDDKK  EUR3M  USD OIS 3m  EUR OIS 3m  \
0 2015-01-31  1.1291  6.5920   9.50      0.1702    -0.12700   
1 2015-02-28  1.1196  6.6648  12.85      0.1672    -0.15375   

   EURUSD basis swap 3m     SPXT  LUATTRUU   MSCIW_H  GLTREAS_H  
0                   NaN  3656.28   2163

## 4. Bloomberg – ugentlig og daglig (robusthed)

In [5]:
fil = raw("bbg_w")

# Ugentlig: fem serier til h* (renterne deles med 52 i analysen, ikke her)
uge_ark = ["EURUSD", "SPXT", "LUATTRUU", "USD OIS 3m", "EUR OIS 3m"]
uge = reduce(lambda l, r: pd.merge(l, r, on="date", how="outer"),
             [les_bbg_ark(fil, a, "W") for a in uge_ark])
uge = uge.sort_values("date").reset_index(drop=True)
print("Ikke-fredage:", (uge["date"].dt.dayofweek != 4).sum(),
      "| manglende værdier:", int(uge.isna().sum().sum()))
uge.to_csv(os.path.join(CLEAN_DIR, "bloomberg_weekly.csv"), index=False)
print("Gemt bloomberg_weekly.csv:", uge.shape, uge["date"].min().date(), "->", uge["date"].max().date())

# Daglig EUR/USD -> månedsgennemsnit
dag = les_bbg_ark(fil, "EURUSD daglig", "D")
dag["month"] = til_md(dag["date"])
gns = dag.groupby("month")["EURUSD daglig"].mean().rename("EURUSD_gns").reset_index()
gem(afgraens(gns), "eurusd_gns_monthly.csv")

Ikke-fredage: 0 | manglende værdier: 0
Gemt bloomberg_weekly.csv: (613, 6) 2015-01-02 -> 2026-09-25
Gemt eurusd_gns_monthly.csv: (141, 2), 2015-01-31 -> 2026-09-30


## 5. GPR (Caldara & Iacoviello)

In [6]:
gpr = pd.read_excel(raw("GPR"))                       # kræver xlrd
gpr = gpr[["month", "GPR", "GPRT", "GPRA", "GPRC_USA"]].dropna(subset=["GPR"]).copy()
gpr["month"] = til_md(gpr["month"])
gem(afgraens(gpr), "gpr_monthly.csv")

Gemt gpr_monthly.csv: (141, 5), 2015-01-31 -> 2026-09-30


## 6. TPU (Caldara et al.)

In [7]:
tpu = pd.read_excel(raw("TPU"), sheet_name="TPU_MONTHLY")[["DATE", "TPU"]].copy()
tpu["month"] = til_md(tpu["DATE"])
gem(afgraens(tpu[["month", "TPU"]]), "tpu_monthly.csv")

Gemt tpu_monthly.csv: (141, 2), 2015-01-31 -> 2026-09-30


## 7. FRED – daglige serier (VIX, dollarindeks, 10-årig rente)

In [8]:
# VIX: månedsgennemsnit
vix = les_fred("VIXCLS", "VIXCLS")
vix["month"] = til_md(vix["date"])
gem(afgraens(vix.groupby("month")["VIXCLS"].mean().reset_index()), "vix_monthly.csv")

# Bredt dollarindeks: værdi ved månedens udgang (som EUR/USD); last() springer tomme dage over
usd = les_fred("USD_INDEX", "DTWEXBGS")
usd["month"] = til_md(usd["date"])
gem(afgraens(usd.groupby("month")["DTWEXBGS"].last().reset_index()), "usd_index_monthly.csv")

# 10-årig rente: månedsgennemsnit
dgs10 = les_fred("DGS10", "DGS10")
dgs10["month"] = til_md(dgs10["date"])
gem(afgraens(dgs10.groupby("month")["DGS10"].mean().reset_index()), "DGS10_monthly.csv")

# 2-årig rente: månedsgennemsnit
dgs2 = les_fred("DGS2", "DGS2")
dgs2["month"] = til_md(dgs2["date"])
gem(afgraens(dgs2.groupby("month")["DGS2"].mean().reset_index()), "DGS2_monthly.csv")

Gemt vix_monthly.csv: (141, 2), 2015-01-31 -> 2026-09-30
Gemt usd_index_monthly.csv: (141, 2), 2015-01-31 -> 2026-09-30
Gemt DGS10_monthly.csv: (141, 2), 2015-01-31 -> 2026-09-30
Gemt DGS2_monthly.csv: (141, 2), 2015-01-31 -> 2026-09-30


# 8. FRED – månedlige serier (kerneinflation, industriproduktion, beskæftigelse)

In [9]:
# Kerneinflation. Oktober 2025 blev ikke offentliggjort (statslukning) -> niveauet interpoleres
cpi = les_fred("CPI", "CPILFESL")
cpi["month"] = til_md(cpi["date"])
cpi = cpi.sort_values("month").reset_index(drop=True)
cpi["cpi_interpoleret"] = cpi["CPILFESL"].isna()
cpi["CPILFESL"] = cpi["CPILFESL"].interpolate(method="linear", limit_area="inside")
cpi["inflation"] = 100 * np.log(cpi["CPILFESL"]).diff()     # måned til måned

print("Interpolerede måneder:", cpi.loc[cpi["cpi_interpoleret"], "month"].dt.date.tolist())
print(cpi.loc[cpi["month"].between("2025-08-31", "2026-01-31"),
              ["month", "CPILFESL", "cpi_interpoleret", "inflation"]])

# afgrænses EFTER beregningen, så januar 2015 bruger december 2014
gem(afgraens(cpi[["month", "inflation"]]), "core_inflation_monthly.csv")

# Industriproduktion og beskæftigelse (niveauer; vækstrater beregnes i LP-notebooken)
for navn, kol, ud in [("INDPRO", "INDPRO", "indpro_monthly.csv"),
                      ("NFP",    "PAYEMS", "nfp_monthly.csv")]:
    df = les_fred(navn, kol)
    df["month"] = til_md(df["date"])
    gem(afgraens(df[["month", kol]]), ud)

Interpolerede måneder: [datetime.date(2025, 10, 31)]
         month  CPILFESL  cpi_interpoleret  inflation
128 2025-08-31  329.7000             False   0.309243
129 2025-09-30  330.4180             False   0.217537
130 2025-10-31  330.7305              True   0.094532
131 2025-11-30  331.0430             False   0.094443
132 2025-12-31  331.8140             False   0.232629
133 2026-01-31  332.7930             False   0.294610
Gemt core_inflation_monthly.csv: (141, 2), 2015-01-31 -> 2026-09-30
Gemt indpro_monthly.csv: (141, 2), 2015-01-31 -> 2026-09-30
Gemt nfp_monthly.csv: (141, 2), 2015-01-31 -> 2026-09-30


## 9. CRSP (slutter december 2025, kun robusthed)

In [10]:
eq = pd.read_csv(raw("crsp_eq"))
eq["month"] = til_md(pd.to_datetime(eq["mthcaldt"], format="%d/%m/%Y"))
gem(afgraens(eq[["month", "vwretd"]]), "us_equity_monthly.csv")

bond = pd.read_csv(raw("crsp_bd"))
bond = bond[(bond["TIDXFAM"] == "BONDMATPORT") &
            (bond["TTERMLBL"] == "Fama Maturity Portfolios - > 60 and <= 120 Month")].copy()
bond["month"] = til_md(pd.to_datetime(bond["MCALDT"], format="%d/%m/%Y"))
bond = bond[["month", "TMEWRETD"]].rename(columns={"TMEWRETD": "bond_ret"})
gem(afgraens(bond), "us_bond_monthly.csv")

Gemt us_equity_monthly.csv: (132, 2), 2015-01-31 -> 2025-12-31
Gemt us_bond_monthly.csv: (132, 2), 2015-01-31 -> 2025-12-31


## 10. Nationalbanken – DNFPVALE (pensionssektorens valutaafdækning, mia. kr.)

In [11]:
dnb = pd.read_csv(raw("DNFPVALE"), sep=";", encoding="latin-1", skiprows=3,
                  header=[0, 1], na_values=[".."])     # ".." = ikke offentliggjort

ccy_map = {"Alle valutaer ekskl. DKK": "ALL", "EUR": "EUR", "USD": "USD"}
post_map = {
    "1. Valutaeksponering i alt": "eksp_total",
    "1.1. Afdækket valutaeksponering": "afd_total",
    "1.1.1. Afdækket til kroner": "afd_dkk",
    "1.1.2. Afdækket til euro": "afd_eur",
    "1.1.3. Afdækket til dollar": "afd_usd",
    "1.1.4. Afdækket til øvrige valutaer": "afd_ovr",
    "1.2. Uafdækket valutaeksponering": "uafd_total",
}

# kolonnenavne: valuta_post (valutaen står kun over første kolonne i hver blok)
new_cols, last_ccy = [], None
for i, (top, bot) in enumerate(dnb.columns):
    top = str(top).strip()
    bot = re.sub(r"\.\d+$", "", str(bot).strip())
    if i == 0:
        new_cols.append("month"); continue
    if top and not top.startswith("Unnamed"):
        last_ccy = ccy_map.get(top, top)
    new_cols.append(f"{last_ccy}_{post_map.get(bot, bot)}")
dnb.columns = new_cols

# behold kun rækker med en måned (2015M01 osv.)
dnb["month"] = dnb["month"].astype(str).str.strip()
dnb = dnb[dnb["month"].str.match(r"^\d{4}M\d{2}$")].copy()
dnb["month"] = til_md(pd.to_datetime(dnb["month"].str.replace("M", "-"), format="%Y-%m"))
tal = dnb.columns.drop("month")
dnb[tal] = dnb[tal].apply(pd.to_numeric, errors="coerce")
dnb = dnb.sort_values("month").reset_index(drop=True)

# hedge ratios = afdækket i alt / eksponering i alt
for c in ["USD", "EUR", "ALL"]:
    dnb[f"{c}_hedge_ratio"] = dnb[f"{c}_afd_total"] / dnb[f"{c}_eksp_total"]

print("Manglende værdier:\n", dnb.isna().sum()[lambda s: s > 0])   # forventet: EUR_afd_usd, EUR_afd_ovr
print(dnb[["month", "USD_eksp_total", "USD_afd_total", "USD_hedge_ratio"]].tail(3))
gem(afgraens(dnb), "dnfpvale_monthly.csv")

Manglende værdier:
 EUR_afd_usd    5
EUR_afd_ovr    5
dtype: int64
         month  USD_eksp_total  USD_afd_total  USD_hedge_ratio
138 2026-07-31         1916.06        1231.30         0.642621
139 2026-08-31         1924.87        1247.74         0.648220
140 2026-09-30         1936.15        1256.17         0.648798
Gemt dnfpvale_monthly.csv: (141, 25), 2015-01-31 -> 2026-09-30


## 11. Effektiv toldsats (Census, som Khalil & Strobel, 2024)

In [12]:
raw_told = pd.read_csv(raw("Toldsatser"), skiprows=1, header=None, index_col=0)
told = raw_told.T                                       # vend: én række pr. måned
told = told[told["Time"].astype(str).str.match(r"^[A-Za-z]+ \d{4}$")].copy()   # kun måneder
told["month"] = til_md(pd.to_datetime(told["Time"], format="%B %Y"))
told["told_beregnet"] = pd.to_numeric(told["Calculated Duty($US)"])
told["import_vaerdi"] = pd.to_numeric(told["Customs Value (Cons) ($US)"])
told["toldsats"] = 100 * told["told_beregnet"] / told["import_vaerdi"]   # i procent
told = afgraens(told[["month", "toldsats", "told_beregnet", "import_vaerdi"]])

print(told.iloc[::12][["month", "toldsats"]].round(2))      # én måned pr. år
print(told[["month", "toldsats"]].tail(6).round(2))

# sæsontjek i den rolige periode 2015–2017
rolig = told[told["month"] < "2018-01-31"]
print(rolig.groupby(rolig["month"].dt.month)["toldsats"].mean().round(3))

gem(told[["month", "toldsats"]], "toldsats_monthly.csv")

0        month  toldsats
0   2015-01-31      1.48
12  2016-01-31      1.62
24  2017-01-31      1.55
36  2018-01-31      1.45
48  2019-01-31      2.55
60  2020-01-31      3.19
72  2021-01-31      3.07
84  2022-01-31      3.11
96  2023-01-31      2.57
108 2024-01-31      2.47
120 2025-01-31      2.22
132 2026-01-31      9.87
0        month  toldsats
134 2026-03-31      6.85
135 2026-04-30      6.74
136 2026-05-31      6.89
137 2026-06-30      6.81
138 2026-07-31      6.45
139 2026-08-31      6.87
month
1     1.551
2     1.522
3     1.360
4     1.373
5     1.414
6     1.457
7     1.587
8     1.616
9     1.522
10    1.487
11    1.397
12    1.397
Name: toldsats, dtype: float64
Gemt toldsats_monthly.csv: (140, 2), 2015-01-31 -> 2026-08-31


## 12. Master – samlet månedligt datasæt

In [13]:
files = [
    "bloomberg_monthly.csv",
    "eurusd_gns_monthly.csv",
    "gpr_monthly.csv",
    "tpu_monthly.csv",
    "vix_monthly.csv",
    "usd_index_monthly.csv",
    "DGS10_monthly.csv",
    "DGS2_monthly.csv",
    "core_inflation_monthly.csv",
    "indpro_monthly.csv",
    "nfp_monthly.csv",
    "us_equity_monthly.csv",
    "us_bond_monthly.csv",
    "dnfpvale_monthly.csv",
    "toldsats_monthly.csv",
]

frames = []
for f in files:
    df = pd.read_csv(os.path.join(CLEAN_DIR, f))
    df["month"] = pd.to_datetime(df["month"])
    frames.append(df)

master = reduce(lambda l, r: pd.merge(l, r, on="month", how="outer"), frames)
master = afgraens(master.sort_values("month"))

# Covid-dummy: 1 i marts–juni 2020 (ekstreme udsving i beskæftigelse og industriproduktion)
master["covid"] = master["month"].between("2020-03-31", "2020-06-30").astype(int)

assert master["month"].is_unique
master.to_csv(os.path.join(CLEAN_DIR, "master_monthly.csv"), index=False)

print("Master:", master.shape)
print("Periode:", master["month"].min().date(), "->", master["month"].max().date())
print("Manglende værdier pr. kolonne:\n", master.isna().sum()[lambda s: s > 0])
print(master.iloc[[0, -1]][["month", "EURUSD", "SPXT", "TPU", "GPR", "inflation", "USD_hedge_ratio", "toldsats"]])

Master: (141, 52)
Periode: 2015-01-31 -> 2026-09-30
Manglende værdier pr. kolonne:
 EURUSD basis swap 3m    73
vwretd                   9
bond_ret                 9
EUR_afd_usd              5
EUR_afd_ovr              5
toldsats                 1
dtype: int64
         month  EURUSD      SPXT         TPU         GPR  inflation  \
0   2015-01-31  1.1291   3656.28   26.612767  111.985695   0.094703   
140 2026-09-30  1.1330  17160.41  260.757130  146.209900   0.334289   

     USD_hedge_ratio  toldsats  
0           0.761942  1.484714  
140         0.648798       NaN  
